In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/ab_data.csv")
print(df.shape)
df.head()

(294478, 5)


,user_id,timestamp,group,landing_page,converted
0,851104,2017-01-21 22:11:48.556739,control,old_page,0
1,804228,2017-01-12 08:01:45.159739,control,old_page,0
2,661590,2017-01-11 16:55:06.154213,treatment,new_page,0
3,853541,2017-01-08 18:28:03.143765,treatment,new_page,0
4,864975,2017-01-21 01:52:26.210827,control,old_page,1


In [2]:
pd.crosstab(df["group"], df["landing_page"])

landing_page,new_page,old_page
group,,
control,1928,145274
treatment,145311,1965


In [3]:
print("Duplicate users:", df["user_id"].duplicated().sum())
print("Missing values:\n", df.isna().sum())
print("Dates:", df["timestamp"].min(), "to", df["timestamp"].max())
print("Overall conversion:", round(df["converted"].mean() * 100, 2), "%")

Duplicate users: 3894
Missing values:
 user_id         0
timestamp       0
group           0
landing_page    0
converted       0
dtype: int64
Dates: 2017-01-02 13:42:05.378582 to 2017-01-24 13:41:54.460509
Overall conversion: 11.97 %


In [4]:
mismatch = (df["group"] == "control") != (df["landing_page"] == "old_page")
print("Mismatched rows:", mismatch.sum())

dup_users = df.loc[df["user_id"].duplicated(keep=False), "user_id"].unique()
print("Users appearing twice:", len(dup_users))
print("...of which have a mismatched row:", df.loc[mismatch, "user_id"].isin(dup_users).sum())

Mismatched rows: 3893
Users appearing twice: 3894
...of which have a mismatched row: 3893


In [5]:
clean = df[~mismatch].copy()
print("After removing mismatches:", clean.shape, "| duplicates left:", clean["user_id"].duplicated().sum())

clean = clean.drop_duplicates(subset="user_id", keep="first")
clean["timestamp"] = pd.to_datetime(clean["timestamp"])
clean["date"] = clean["timestamp"].dt.date

print("Final:", clean.shape)
print(clean["group"].value_counts())

import os
os.makedirs("data/clean", exist_ok=True)
clean.to_csv("data/clean/ab_clean.csv", index=False)

After removing mismatches: (290585, 5) | duplicates left: 1
Final: (290584, 6)
group
treatment    145310
control      145274
Name: count, dtype: int64
